<a href="https://colab.research.google.com/github/hassan1662n/Deep-learning-projects/blob/main/Sentiment_Analysis_on_IMDB_Reviews_with_LSTM_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
kaggle_dictionary = json.load(open('kaggle.json'))
os.environ['KAGGLE_USERNAME'] = kaggle_dictionary['username']
os.environ['KAGGLE_KEY'] = kaggle_dictionary['key']

In [9]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:00<00:00, 156MB/s] 



In [10]:
with ZipFile('imdb-dataset-of-50k-movie-reviews.zip', 'r') as zipObj:
    zipObj.extractall()

In [11]:
df = pd.read_csv('IMDB Dataset.csv')

In [12]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [14]:
df.shape

(50000, 2)

In [13]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [15]:
df.replace({'positive': 1, 'negative': 0}, inplace=True)

/tmp/ipykernel_985/1160970408.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'positive': 1, 'negative': 0}, inplace=True)


In [16]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [17]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

In [18]:
Tokenizer = Tokenizer(num_words=5000)
Tokenizer.fit_on_texts(train_data['review'])
X_train = pad_sequences(Tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test = pad_sequences(Tokenizer.texts_to_sequences(test_data['review']), maxlen=200)

In [20]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [21]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

In [29]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [31]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [33]:
history = model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_data=(X_test, Y_test))

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 349s 546ms/step - accuracy: 0.7835 - loss: 0.4671 - val_accuracy: 0.8198 - val_loss: 0.4081
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 309s 494ms/step - accuracy: 0.8436 - loss: 0.3689 - val_accuracy: 0.8594 - val_loss: 0.3642
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 305s 487ms/step - accuracy: 0.8726 - loss: 0.3121 - val_accuracy: 0.8685 - val_loss: 0.3216
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 321s 486ms/step - accuracy: 0.8749 - loss: 0.3017 - val_accuracy: 0.8866 - val_loss: 0.2830
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 307s 492ms/step - accuracy: 0.9018 - loss: 0.2424 - val_accuracy: 0.8628 - val_loss: 0.3184


In [34]:
loss , accuracy = model.evaluate(X_test, Y_test)
print(f'Loss: {loss}')
print(f'Accuracy: {accuracy}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 150ms/step - accuracy: 0.8628 - loss: 0.3184
Loss: 0.3184496760368347
Accuracy: 0.8628000020980835


In [35]:
def predict_sentiment(text):
    sequence = Tokenizer.texts_to_sequences([text])
    sequence = pad_sequences(sequence, maxlen=200)
    prediction = model.predict(sequence)[0]
    return 'positive' if prediction > 0.5 else 'negative'

In [37]:
new_review = "this movie was good i loved it"
predicted_sentiment = predict_sentiment(new_review)
print(f"Predicted sentiment: {predicted_sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
Predicted sentiment: positive


In [36]:
new_review = "this movie was not good, what a waste of money"
predicted_sentiment = predict_sentiment(new_review)
print(f"Predicted sentiment: {predicted_sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 454ms/step
Predicted sentiment: negative
